In [ ]:
import pandas as pd
from datasets import Dataset, DatasetDict
from collections import Counter
from huggingface_hub import login

login(token="")

/projects/extern/kisski/kisski_tegami/dir.project/micromamba/envs/jupyter/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [15]:
df = pd.read_csv("raw/edos_labelled_aggregated.csv")
df = df[["rewire_id", "text", "label_sexist", "split"]]

# Convert label text to uppercase
df["label_text"] = df["label_sexist"].str.upper()

# Map labels to integers
mapping = {"sexist": 1, "not sexist": 0}
df["label"] = df["label_sexist"].map(mapping)
df = df.drop("label_sexist", axis=1)

df.head()


,rewire_id,text,split,label_text,label
0,sexism2022_english-9609,"In Nigeria, if you rape a woman, the men rape ...",dev,NOT SEXIST,0
1,sexism2022_english-16993,"Then, she's a keeper. 😉",train,NOT SEXIST,0
2,sexism2022_english-13149,This is like the Metallica video where the poo...,train,NOT SEXIST,0
3,sexism2022_english-13021,woman?,train,NOT SEXIST,0
4,sexism2022_english-966,I bet she wished she had a gun,dev,NOT SEXIST,0


In [17]:
# asess current split
print("Train:", len(df[df.split == "train"]) / len(df))
print("Test:", len(df[df.split == "test"]) / len(df))
print("Eval:", len(df[df.split == "dev"]) / len(df))

Train: 0.7
Test: 0.2
Eval: 0.1


In [10]:
dd = DatasetDict({
    "train": Dataset.from_pandas(df[df.split == "train"].drop("split", axis=1), preserve_index=False),
    "validation": Dataset.from_pandas(df[df.split == "dev"].drop("split", axis=1), preserve_index=False),
    "test": Dataset.from_pandas(df[df.split == "test"].drop("split", axis=1), preserve_index=False)
})

dd

DatasetDict({
    train: Dataset({
        features: ['rewire_id', 'text', 'label_text', 'label'],
        num_rows: 14000
    })
    validation: Dataset({
        features: ['rewire_id', 'text', 'label_text', 'label'],
        num_rows: 2000
    })
    test: Dataset({
        features: ['rewire_id', 'text', 'label_text', 'label'],
        num_rows: 4000
    })
})

In [11]:
# Inspect distributions
for k, v in dd.items():
    print(f"=== {k} ===")
    counts = Counter(v["label_text"])
    total = sum(counts.values())
    
    for label, count in counts.items():
        percent = (count / total) * 100
        print(f"{label}: {count} ({percent:.2f}%)")
    
    print()


=== train ===
NOT SEXIST: 10602 (75.73%)
SEXIST: 3398 (24.27%)

=== validation ===
NOT SEXIST: 1514 (75.70%)
SEXIST: 486 (24.30%)

=== test ===
NOT SEXIST: 3030 (75.75%)
SEXIST: 970 (24.25%)



In [6]:
# Push to HF
dd.push_to_hub("TomData/EDOS", private=True)

Creating parquet from Arrow format: 100%|██████████| 14/14 [00:00<00:00, 391.56ba/s]
Uploading files as a binary IO buffer is not supported by Xet Storage. Falling back to HTTP upload.
Creating parquet from Arrow format: 100%|██████████| 2/2 [00:00<00:00, 249.73ba/s]
Uploading files as a binary IO buffer is not supported by Xet Storage. Falling back to HTTP upload.
Creating parquet from Arrow format: 100%|██████████| 4/4 [00:00<00:00, 445.92ba/s]
Uploading files as a binary IO buffer is not supported by Xet Storage. Falling back to HTTP upload.
Uploading the dataset shards: 100%|██████████| 1/1 [00:02<00:00,  2.49s/it]


CommitInfo(commit_url='https://huggingface.co/datasets/TomData/EDOS/commit/25c51196f5ecd41bd1bc150d1a62e83d3f853c73', commit_message='Upload dataset', commit_description='', oid='25c51196f5ecd41bd1bc150d1a62e83d3f853c73', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/TomData/EDOS', endpoint='https://huggingface.co', repo_type='dataset', repo_id='TomData/EDOS'), pr_revision=None, pr_num=None)